Task 01 | Nested Transaction Analyzer
Concepts: Data structures, sorting, comprehensions


In [20]:
REQUIRED_KEYS = {"id", "customer", "category", "amount", "status"}


def analyze_transactions(transactions):
    """Return a report: customer, total successful spending, top category, rank."""
    if not isinstance(transactions, list):
        raise TypeError("transactions must be a list of dictionaries")

    seen_ids = set()
    totals = {}        # customer -> total successful spending
    by_category = {}   # customer -> {category: amount}

    for t in transactions:
        if not isinstance(t, dict) or not REQUIRED_KEYS <= t.keys():
            raise ValueError(f"Invalid transaction: {t!r}")
        if t["id"] in seen_ids:
            raise ValueError(f"Duplicate transaction id: {t['id']}")
        seen_ids.add(t["id"])

        customer = t["customer"]
        # every customer appears in the report, even with only failed transactions
        totals.setdefault(customer, 0)
        by_category.setdefault(customer, {})

        if t["status"] != "success":
            continue  # ignore failed transactions

        amount = t["amount"]
        if isinstance(amount, bool) or not isinstance(amount, (int, float)) or amount < 0:
            raise ValueError(f"Invalid amount in transaction {t['id']}: {amount!r}")

        totals[customer] += amount
        cats = by_category[customer]
        cats[t["category"]] = cats.get(t["category"], 0) + amount

    report = []
    for customer, total in totals.items():
        cats = by_category[customer]
        # highest amount wins; ties -> alphabetical category
        top = min(cats, key=lambda c: (-cats[c], c)) if cats else None
        report.append({"customer": customer, "total": total, "top_category": top})

    # spending descending, then customer name ascending
    report.sort(key=lambda r: (-r["total"], r["customer"]))

    # rank (equal spending shares the same rank)
    rank, previous_total = 0, None
    for position, row in enumerate(report, start=1):
        if row["total"] != previous_total:
            rank, previous_total = position, row["total"]
        row["rank"] = rank
    return report


if __name__ == "__main__":
    data = [
        {"id": 1, "customer": "Ali", "category": "Food", "amount": 500, "status": "success"},
        {"id": 2, "customer": "Ali", "category": "Travel", "amount": 200, "status": "failed"},
        {"id": 3, "customer": "Sara", "category": "Food", "amount": 700, "status": "success"},
        {"id": 4, "customer": "Zain", "category": "Food", "amount": 50, "status": "failed"},
    ]
    for row in analyze_transactions(data):
        print(row)
    try:
        analyze_transactions(data + [data[0]])
    except ValueError as err:
        print("Rejected:", err)


{'customer': 'Sara', 'total': 700, 'top_category': 'Food', 'rank': 1}
{'customer': 'Ali', 'total': 500, 'top_category': 'Food', 'rank': 2}
{'customer': 'Zain', 'total': 0, 'top_category': None, 'rank': 3}
Rejected: Duplicate transaction id: 1


Task 02 | Longest Valid Window<br>
Concepts: Sliding window, dictionaries

In [19]:
def longest_unique_window(text):
    """Return (substring, start_index, length) of the longest window with no repeated characters."""
    if not isinstance(text, str):
        raise TypeError("text must be a string")

    last_seen = {}            # character -> last index where it appeared
    window_start = 0
    best_start, best_length = 0, 0

    for i, ch in enumerate(text):
        # if ch is already inside the current window, move the window start past it
        if ch in last_seen and last_seen[ch] >= window_start:
            window_start = last_seen[ch] + 1
        last_seen[ch] = i

        length = i - window_start + 1
        if length > best_length:   # strictly greater -> earliest window wins ties
            best_start, best_length = window_start, length

    return text[best_start:best_start + best_length], best_start, best_length


if __name__ == "__main__":
    for sample in ["abcabcbb", "", "bbbbb", "pwwkew", "a b!a b"]:
        print(repr(sample), "->", longest_unique_window(sample))


'abcabcbb' -> ('abc', 0, 3)
'' -> ('', 0, 0)
'bbbbb' -> ('b', 0, 1)
'pwwkew' -> ('wke', 2, 3)
'a b!a b' -> ('a b!', 0, 4)


Task 03 | Recursive JSON Path Explorer<br>
Concepts: Recursion, nested lists/dictionaries


In [21]:
def _same_value(value, target):
    """Equality that does not treat True/False as the numbers 1/0."""
    if isinstance(value, bool) or isinstance(target, bool):
        return type(value) is type(target) and value == target
    return value == target


def find_paths(data, target, _prefix=""):
    """Return all paths (dot notation for keys, [i] for list items) whose value equals target."""
    paths = []

    if _prefix and _same_value(data, target):
        paths.append(_prefix)

    if isinstance(data, dict):
        for key, value in data.items():
            path = f"{_prefix}.{key}" if _prefix else str(key)
            paths.extend(find_paths(value, target, path))
    elif isinstance(data, list):
        for index, value in enumerate(data):
            paths.extend(find_paths(value, target, f"{_prefix}[{index}]"))

    return paths


if __name__ == "__main__":
    sample = {"a": [{"x": 5}, 5], "b": {"y": 5}}
    print(find_paths(sample, 5))                      # ['a[0].x', 'a[1]', 'b.y']
    print(find_paths({"p": True, "q": 1}, 1))         # ['q']  (True is not 1)
    print(find_paths({"p": True, "q": 1}, True))      # ['p']
    print(find_paths([[1, [1]], {"k": [1]}], 1))      # ['[0][0]', '[0][1][0]', '[1].k[0]']
    print(find_paths({}, 5))                          # []


['a[0].x', 'a[1]', 'b.y']
['q']
['p']
['[0][0]', '[0][1][0]', '[1].k[0]']
[]


Task 04 | Dependency Execution Planner<br>
Concepts: Graphs, sets, algorithms


In [22]:
import heapq


def plan_execution(tasks, dependencies):
    """Return a valid execution order. dependencies = [(A, B)] means A must finish before B."""
    tasks = list(tasks)
    if len(set(tasks)) != len(tasks):
        raise ValueError("Duplicate task names are not allowed")
    known = set(tasks)

    followers = {t: set() for t in tasks}   # task -> tasks that wait for it
    indegree = {t: 0 for t in tasks}        # number of unfinished prerequisites

    for pair in dependencies:
        if not isinstance(pair, (tuple, list)) or len(pair) != 2:
            raise ValueError(f"Invalid dependency: {pair!r}")
        first, second = pair
        for name in (first, second):
            if name not in known:
                raise ValueError(f"Unknown task name: {name!r}")
        if first == second:
            raise ValueError(f"Cycle detected: task {first!r} depends on itself")
        if second not in followers[first]:
            followers[first].add(second)
            indegree[second] += 1

    # min-heap -> the alphabetically first available task is always chosen
    available = [t for t in tasks if indegree[t] == 0]
    heapq.heapify(available)

    order = []
    while available:
        current = heapq.heappop(available)
        order.append(current)
        for nxt in followers[current]:
            indegree[nxt] -= 1
            if indegree[nxt] == 0:
                heapq.heappush(available, nxt)

    if len(order) != len(tasks):
        stuck = sorted(t for t in tasks if indegree[t] > 0)
        raise ValueError(f"Cycle detected involving tasks: {', '.join(map(str, stuck))}")
    return order


if __name__ == "__main__":
    print(plan_execution(["build", "test", "deploy"],
                         [("build", "test"), ("test", "deploy")]))
    print(plan_execution(["c", "b", "a"], []))   # ['a', 'b', 'c']
    for bad in ([("a", "b"), ("b", "a")], [("a", "zzz")]):
        try:
            plan_execution(["a", "b"], bad)
        except ValueError as err:
            print("Error:", err)


['build', 'test', 'deploy']
['a', 'b', 'c']
Error: Cycle detected involving tasks: a, b
Error: Unknown task name: 'zzz'


Task 05 | Expression Validator and Evaluator<br>
Concepts: Stacks, parsing, exception handling


In [23]:
PRECEDENCE = {"+": 1, "-": 1, "*": 2, "/": 2, "neg": 3}   # "neg" = unary minus


def tokenize(expression):
    """Turn the text into a list of ints and operator/parenthesis characters."""
    tokens, i = [], 0
    while i < len(expression):
        ch = expression[i]
        if ch.isspace():
            i += 1
        elif ch in "0123456789":
            j = i
            while j < len(expression) and expression[j] in "0123456789":
                j += 1
            tokens.append(int(expression[i:j]))
            i = j
        elif ch in "+-*/()":
            tokens.append(ch)
            i += 1
        else:
            raise ValueError(f"Invalid character {ch!r} at position {i}")
    return tokens


def _apply(operator, values):
    if operator == "neg":
        values[-1] = -values[-1]
        return
    right = values.pop()
    left = values.pop()
    if operator == "+":
        values.append(left + right)
    elif operator == "-":
        values.append(left - right)
    elif operator == "*":
        values.append(left * right)
    else:
        if right == 0:
            raise ZeroDivisionError("Division by zero")
        values.append(left / right)


def evaluate(expression):
    """Evaluate an integer expression with + - * / parentheses and unary minus."""
    if not isinstance(expression, str):
        raise TypeError("expression must be a string")
    tokens = tokenize(expression)
    if not tokens:
        raise ValueError("Empty expression")

    values, operators = [], []
    expect_operand = True    # True when a number, '(' or unary sign must come next

    for token in tokens:
        if isinstance(token, int):
            if not expect_operand:
                raise ValueError("Missing operator before number")
            values.append(token)
            expect_operand = False
        elif token == "(":
            if not expect_operand:
                raise ValueError("Missing operator before '('")
            operators.append("(")
        elif token == ")":
            if expect_operand:
                raise ValueError("Unexpected ')'")
            while operators and operators[-1] != "(":
                _apply(operators.pop(), values)
            if not operators:
                raise ValueError("Unbalanced parentheses")
            operators.pop()          # remove '('
        elif expect_operand:         # a sign in front of an operand
            if token == "-":
                operators.append("neg")
            elif token != "+":
                raise ValueError(f"Unexpected operator {token!r}")
        else:                        # binary operator
            while (operators and operators[-1] != "("
                   and PRECEDENCE[operators[-1]] >= PRECEDENCE[token]):
                _apply(operators.pop(), values)
            operators.append(token)
            expect_operand = True

    if expect_operand:
        raise ValueError("Expression ends unexpectedly")
    while operators:
        operator = operators.pop()
        if operator == "(":
            raise ValueError("Unbalanced parentheses")
        _apply(operator, values)
    return values[0]


if __name__ == "__main__":
    tests = ["-(3 + 2) * 4 + 18 / 3", "2 + 3 * 4", "(2 + 3) * 4", "2 * -3", "10 / 0",
             "2 +", "(1 + 2", "1 + 2)", "3 $ 4", "", "4 5", "()"]
    for text in tests:
        try:
            print(f"{text!r:28} = {evaluate(text)}")
        except (ValueError, ZeroDivisionError) as err:
            print(f"{text!r:28} error: {err}")


'-(3 + 2) * 4 + 18 / 3'      = -14.0
'2 + 3 * 4'                  = 14
'(2 + 3) * 4'                = 20
'2 * -3'                     = -6
'10 / 0'                     error: Division by zero
'2 +'                        error: Expression ends unexpectedly
'(1 + 2'                     error: Unbalanced parentheses
'1 + 2)'                     error: Unbalanced parentheses
'3 $ 4'                      error: Invalid character '$' at position 2
''                           error: Empty expression
'4 5'                        error: Missing operator before number
'()'                         error: Unexpected ')'


Task 06 | Banking Ledger with OOP<br>
Concepts: Classes, encapsulation, properties


In [24]:
import itertools
from dataclasses import dataclass

_transaction_ids = itertools.count(1)


class InsufficientFundsError(Exception):
    pass


class AccountNotFoundError(LookupError):
    pass


def _check_amount(amount):
    if isinstance(amount, bool) or not isinstance(amount, (int, float)) or amount <= 0:
        raise ValueError(f"Amount must be a positive number, got {amount!r}")


@dataclass(frozen=True)          # frozen -> records cannot be changed
class Transaction:
    transaction_id: int
    action: str
    amount: float
    balance_after: float


class Account:
    def __init__(self, account_id, owner, opening_balance=0):
        if isinstance(opening_balance, bool) or not isinstance(opening_balance, (int, float)) \
                or opening_balance < 0:
            raise ValueError("Opening balance cannot be negative")
        self._account_id = account_id
        self._owner = owner
        self._balance = 0
        self._history = []
        if opening_balance > 0:
            self.deposit(opening_balance, action="opening")

    @property
    def account_id(self):
        return self._account_id

    @property
    def owner(self):
        return self._owner

    @property
    def balance(self):           # read-only: balance only changes via deposit/withdraw
        return self._balance

    def _record(self, action, amount):
        self._history.append(Transaction(next(_transaction_ids), action, amount, self._balance))

    def deposit(self, amount, action="deposit"):
        _check_amount(amount)
        self._balance += amount
        self._record(action, amount)

    def withdraw(self, amount, action="withdraw"):
        _check_amount(amount)
        if amount > self._balance:
            raise InsufficientFundsError(
                f"Account {self._account_id}: balance {self._balance} is less than {amount}")
        self._balance -= amount
        self._record(action, amount)

    def statement(self):
        return tuple(self._history)


class Bank:
    def __init__(self):
        self._accounts = {}

    def open_account(self, account_id, owner, opening_balance=0):
        if account_id in self._accounts:
            raise ValueError(f"Account {account_id} already exists")
        account = Account(account_id, owner, opening_balance)
        self._accounts[account_id] = account
        return account

    def get_account(self, account_id):
        try:
            return self._accounts[account_id]
        except KeyError:
            raise AccountNotFoundError(f"Account {account_id} does not exist") from None

    def deposit(self, account_id, amount):
        self.get_account(account_id).deposit(amount)

    def withdraw(self, account_id, amount):
        self.get_account(account_id).withdraw(amount)

    def transfer(self, from_id, to_id, amount):
        """Atomic: every check is done first, so either both balances change or neither."""
        source = self.get_account(from_id)
        target = self.get_account(to_id)
        _check_amount(amount)
        if source is target:
            raise ValueError("Cannot transfer to the same account")
        if amount > source.balance:
            raise InsufficientFundsError(
                f"Account {from_id}: balance {source.balance} is less than {amount}")
        source.withdraw(amount, action="transfer_out")
        target.deposit(amount, action="transfer_in")

    def statement(self, account_id):
        return self.get_account(account_id).statement()


if __name__ == "__main__":
    bank = Bank()
    bank.open_account("A", "Ali", 1000)
    bank.open_account("B", "Sara", 200)
    bank.transfer("A", "B", 350)
    print("A =", bank.get_account("A").balance, "| B =", bank.get_account("B").balance)

    for action in (lambda: bank.transfer("A", "B", 5000),
                   lambda: bank.transfer("A", "Z", 10),
                   lambda: bank.deposit("A", -5)):
        try:
            action()
        except (InsufficientFundsError, AccountNotFoundError, ValueError) as err:
            print("Rejected:", err)
    print("After failures -> A =", bank.get_account("A").balance,
          "| B =", bank.get_account("B").balance)

    print("Statement for A:")
    for record in bank.statement("A"):
        print("  ", record)


A = 650 | B = 550
Rejected: Account A: balance 650 is less than 5000
Rejected: Account Z does not exist
Rejected: Amount must be a positive number, got -5
After failures -> A = 650 | B = 550
Statement for A:
   Transaction(transaction_id=1, action='opening', amount=1000, balance_after=1000)
   Transaction(transaction_id=3, action='transfer_out', amount=350, balance_after=650)


Task 07 | Employee Payroll Engine<br>
Concepts: Inheritance, polymorphism, abstract classes

In [25]:
from abc import ABC, abstractmethod


def _non_negative(value, label):
    if isinstance(value, bool) or not isinstance(value, (int, float)) or value < 0:
        raise ValueError(f"{label} must be a non-negative number, got {value!r}")
    return value


class Employee(ABC):
    """Abstract base class - Employee() cannot be created directly."""

    def __init__(self, name):
        if not isinstance(name, str) or not name.strip():
            raise ValueError("Employee name must be a non-empty string")
        self.name = name.strip()

    @abstractmethod
    def calculate_pay(self):
        """Return the pay for one period."""

    def __str__(self):
        return f"{self.__class__.__name__}({self.name})"


class SalariedEmployee(Employee):
    def __init__(self, name, salary):
        super().__init__(name)
        self.salary = _non_negative(salary, "Salary")

    def calculate_pay(self):
        return self.salary


class HourlyEmployee(Employee):
    OVERTIME_AFTER = 40
    OVERTIME_RATE = 1.5

    def __init__(self, name, hourly_rate, hours_worked):
        super().__init__(name)
        self.hourly_rate = _non_negative(hourly_rate, "Hourly rate")
        self.hours_worked = _non_negative(hours_worked, "Hours worked")

    def calculate_pay(self):
        regular = min(self.hours_worked, self.OVERTIME_AFTER)
        overtime = max(self.hours_worked - self.OVERTIME_AFTER, 0)
        return regular * self.hourly_rate + overtime * self.hourly_rate * self.OVERTIME_RATE


class CommissionEmployee(Employee):
    def __init__(self, name, base_pay, sales, rate):
        super().__init__(name)
        self.base_pay = _non_negative(base_pay, "Base pay")
        self.sales = _non_negative(sales, "Sales")
        self.rate = _non_negative(rate, "Commission rate")

    def calculate_pay(self):
        return self.base_pay + self.sales * self.rate


def payroll_summary(employees):
    """Unified payroll summary: one line per employee plus the total."""
    lines, total = [], 0
    for employee in employees:
        if not isinstance(employee, Employee):
            raise TypeError(f"{employee!r} is not an Employee")
        pay = employee.calculate_pay()          # polymorphism: each class has its own formula
        lines.append({"name": employee.name, "type": type(employee).__name__, "pay": pay})
        total += pay
    return {"lines": lines, "total": total}


if __name__ == "__main__":
    try:
        Employee("Nobody")
    except TypeError as err:
        print("Cannot instantiate:", err)

    staff = [SalariedEmployee("Ayesha", 3000),
             HourlyEmployee("Bilal", 20, 45),
             CommissionEmployee("Chand", 500, 4000, 0.10)]
    summary = payroll_summary(staff)
    for line in summary["lines"]:
        print(f"{line['name']:8} {line['type']:20} {line['pay']:>8.2f}")
    print("Total payroll:", summary["total"])


Cannot instantiate: Can't instantiate abstract class Employee without an implementation for abstract method 'calculate_pay'
Ayesha   SalariedEmployee      3000.00
Bilal    HourlyEmployee         950.00
Chand    CommissionEmployee     900.00
Total payroll: 4850.0


Task 08 | Cache Decorator with Expiration<br>
Concepts: Decorators, closures, functools, time


In [27]:
import functools
import time


def ttl_cache(seconds):
    """
    Cache a function's results for `seconds` seconds.

    - Keyword argument order doesn't matter: f(x=2, y=3) and f(y=3, x=2) share one entry.
    - Unhashable arguments (lists, dicts, sets): the result is NOT cached.
      The function runs normally every time and the call counts as a miss.
    """
    def decorator(func):
        cache = {}                       # key -> (result, expires_at)
        hits = misses = 0

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            nonlocal hits, misses
            key = (args, tuple(sorted(kwargs.items())))

            try:
                hash(key)
            except TypeError:            # unhashable input -> skip the cache
                misses += 1
                return func(*args, **kwargs)

            now = time.monotonic()
            if key in cache:
                result, expires_at = cache[key]
                if now < expires_at:
                    hits += 1
                    return result

            misses += 1
            result = func(*args, **kwargs)
            cache[key] = (result, now + seconds)
            return result

        wrapper.cache_info = lambda: {"hits": hits, "misses": misses}
        return wrapper
    return decorator


# ---------------- Test ----------------
@ttl_cache(seconds=1)
def add(x, y):
    """Add two numbers."""
    print(f"  computing {x} + {y}")
    return x + y

print(add(x=2, y=3))      # miss
print(add(y=3, x=2))      # hit: same arguments, different order
print(add.cache_info())

time.sleep(1.1)
print(add(x=2, y=3))      # miss: expired, recomputed
print(add.cache_info())

print(add([1], [2]))      # unhashable: not cached
print(add.cache_info())

print(add.__name__, "-", add.__doc__)

  computing 2 + 3
5
5
{'hits': 1, 'misses': 1}
  computing 2 + 3
5
{'hits': 1, 'misses': 2}
  computing [1] + [2]
[1, 2]
{'hits': 1, 'misses': 3}
add - Add two numbers.


Task 09 | Streaming Log Inspector<br>
Concepts: Generators, iterators, file handling


In [28]:
from collections import Counter
from datetime import datetime


def read_errors(path, stats=None):
    """
    Stream a log file and yield (line_number, timestamp, message) for ERROR lines.

    Expected format: YYYY-MM-DD HH:MM:SS | LEVEL | message
    Malformed lines are skipped and counted in stats["malformed"] if a dict is passed.
    Blank lines are ignored.
    """
    if stats is not None:
        stats.setdefault("malformed", 0)

    with open(path, encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):    # one line in memory at a time
            line = line.strip()
            if not line:
                continue

            parts = line.split(" | ", 2)                    # max 3 parts: message may contain " | "
            try:
                timestamp, level, message = (p.strip() for p in parts)
                datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S")
                if not level or not message:
                    raise ValueError
            except ValueError:
                if stats is not None:
                    stats["malformed"] += 1
                continue

            if level == "ERROR":
                yield line_number, timestamp, message


def summarize_errors(path):
    """Count ERROR records per date. Returns (counts_by_date, malformed_count)."""
    stats = {}
    counts = Counter(timestamp[:10] for _, timestamp, _ in read_errors(path, stats))
    return dict(sorted(counts.items())), stats["malformed"]


# ---------------- Test ----------------
sample_log = """2026-11-07 10:30:00 | ERROR | Connection lost
2026-11-07 10:31:00 | INFO | Reconnected
2026-11-07 11:00:00 | ERROR | Disk full | path=/var/data
this line is garbage

2026-13-40 09:00:00 | ERROR | Bad date
2026-11-08 08:15:00 | WARNING | High memory
2026-11-08 09:00:00 | ERROR | Timeout calling API
2026-11-08 09:05:00 | ERROR
2026-11-09 12:00:00 | ERROR | Job failed
"""
with open("app.log", "w", encoding="utf-8") as f:
    f.write(sample_log)

stats = {}
for record in read_errors("app.log", stats):
    print(record)
print("Malformed lines:", stats["malformed"])

print()
counts, malformed = summarize_errors("app.log")
print("Errors by date:", counts)
print("Malformed lines:", malformed)

(1, '2026-11-07 10:30:00', 'Connection lost')
(3, '2026-11-07 11:00:00', 'Disk full | path=/var/data')
(8, '2026-11-08 09:00:00', 'Timeout calling API')
(10, '2026-11-09 12:00:00', 'Job failed')
Malformed lines: 3

Errors by date: {'2026-11-07': 2, '2026-11-08': 1, '2026-11-09': 1}
Malformed lines: 3


Task 10 | CSV Inventory Reconciliation<br>
Concepts: CSV, file handling, validation


In [30]:
"""Task 10 - CSV Inventory Reconciliation"""
import csv


def load_inventory(path):
    """Read inventory.csv (sku,name,quantity) -> ordered dict {sku: {"name":..., "quantity":...}}."""
    inventory = {}
    with open(path, newline="", encoding="utf-8") as handle:
        for line_number, row in enumerate(csv.DictReader(handle), start=2):
            sku = (row.get("sku") or "").strip()
            name = (row.get("name") or "").strip()
            try:
                quantity = int((row.get("quantity") or "").strip())
            except ValueError:
                raise ValueError(f"{path} line {line_number}: quantity must be an integer")
            if not sku or quantity < 0 or sku in inventory:
                raise ValueError(f"{path} line {line_number}: invalid or duplicate SKU/quantity")
            inventory[sku] = {"name": name, "quantity": quantity}
    return inventory


def apply_movements(inventory, movements_path):
    """Apply IN/OUT movements in order. Returns the list of rejected rows (with a reason)."""
    rejected = []
    with open(movements_path, newline="", encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            sku = (row.get("sku") or "").strip()
            kind = (row.get("type") or "").strip().upper()
            units_text = (row.get("units") or "").strip()
            reason = None
            units = 0

            if sku not in inventory:
                reason = "unknown SKU"
            elif kind not in ("IN", "OUT"):
                reason = "invalid movement type"
            else:
                try:
                    units = int(units_text)
                except ValueError:
                    reason = "units is not an integer"
                else:
                    if units <= 0:
                        reason = "units must be positive"
                    elif kind == "OUT" and units > inventory[sku]["quantity"]:
                        reason = "insufficient stock (would go negative)"

            if reason:
                rejected.append({"sku": sku, "type": row.get("type", ""),
                                 "units": units_text, "reason": reason})
                continue                       # keep processing the next rows
            inventory[sku]["quantity"] += units if kind == "IN" else -units
    return rejected


def write_outputs(inventory, rejected, inventory_out="updated_inventory.csv",
                  rejected_out="rejected_movements.csv"):
    with open(inventory_out, "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["sku", "name", "quantity"])
        for sku, item in inventory.items():
            writer.writerow([sku, item["name"], item["quantity"]])
    with open(rejected_out, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["sku", "type", "units", "reason"])
        writer.writeheader()
        writer.writerows(rejected)


def create_sample_data():
    with open("inventory.csv", "w", newline="", encoding="utf-8") as handle:
        handle.write("sku,name,quantity\nP01,Widget,10\nP02,Gadget,5\n")
    with open("movements.csv", "w", newline="", encoding="utf-8") as handle:
        handle.write("sku,type,units\n"
                     "P01,OUT,4\nP01,OUT,9\nP01,IN,2\n"      # P01 -> 8, second row rejected
                     "P02,IN,0\nP99,IN,3\nP02,MOVE,1\nP02,OUT,abc\nP02,OUT,5\n")


def main():
    create_sample_data()
    inventory = load_inventory("inventory.csv")
    rejected = apply_movements(inventory, "movements.csv")
    write_outputs(inventory, rejected)
    for sku, item in inventory.items():
        print(sku, item["name"], item["quantity"])
    print("Rejected rows:")
    for row in rejected:
        print("  ", row)


if __name__ == "__main__":
    main()


P01 Widget 8
P02 Gadget 0
Rejected rows:
   {'sku': 'P01', 'type': 'OUT', 'units': '9', 'reason': 'insufficient stock (would go negative)'}
   {'sku': 'P02', 'type': 'IN', 'units': '0', 'reason': 'units must be positive'}
   {'sku': 'P99', 'type': 'IN', 'units': '3', 'reason': 'unknown SKU'}
   {'sku': 'P02', 'type': 'MOVE', 'units': '1', 'reason': 'invalid movement type'}
   {'sku': 'P02', 'type': 'OUT', 'units': 'abc', 'reason': 'units is not an integer'}


Task 11 | Multi-File Student Gradebook<br>
Concepts: File handling, dictionaries, exceptions


In [31]:
import csv


def grade_for(average):
    if average >= 85:
        return "A"
    if average >= 70:
        return "B"
    if average >= 55:
        return "C"
    if average >= 40:
        return "D"
    return "F"


def load_students(path):
    """students.csv (student_id,name) -> {student_id: name}. Duplicate IDs are rejected."""
    students = {}
    with open(path, newline="", encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            student_id = (row.get("student_id") or "").strip()
            name = (row.get("name") or "").strip()
            if not student_id or not name:
                raise ValueError(f"Invalid student row: {row}")
            if student_id in students:
                raise ValueError(f"Duplicate student id: {student_id}")
            students[student_id] = name
    return students


def load_scores(path, students):
    """scores.csv -> ({student_id: {subject: marks}}, rejected_rows)."""
    scores, rejected = {}, []
    with open(path, newline="", encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            student_id = (row.get("student_id") or "").strip()
            subject = (row.get("subject") or "").strip()
            reason = None
            marks = None
            try:
                marks = float((row.get("marks") or "").strip())
            except ValueError:
                reason = "marks is not a number"

            if student_id not in students:
                reason = "unknown student id"
            elif not subject:
                reason = "missing subject"
            elif reason is None and not 0 <= marks <= 100:
                reason = "marks must be between 0 and 100"
            elif reason is None and subject in scores.get(student_id, {}):
                reason = "duplicate student-subject entry"

            if reason:
                rejected.append({**row, "reason": reason})
                continue
            scores.setdefault(student_id, {})[subject] = marks
    return scores, rejected


def build_report(students, scores):
    """Average, grade and rank per student (only students who have valid scores)."""
    rows = []
    for student_id, subjects in scores.items():
        average = sum(subjects.values()) / len(subjects)
        rows.append({"student_id": student_id, "name": students[student_id],
                     "subjects": len(subjects), "average": average})
    rows.sort(key=lambda r: (-r["average"], r["student_id"]))   # average desc, then ID
    for rank, row in enumerate(rows, start=1):
        row["rank"] = rank
        row["grade"] = grade_for(row["average"])
        row["average"] = round(row["average"], 2)
    return rows


def write_report(rows, path="report.csv"):
    fields = ["rank", "student_id", "name", "subjects", "average", "grade"]
    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows({f: row[f] for f in fields} for row in rows)


def create_sample_data():
    with open("students.csv", "w", newline="", encoding="utf-8") as handle:
        handle.write("student_id,name\n101,Ali\n102,Sara\n103,Bilal\n")
    with open("scores.csv", "w", newline="", encoding="utf-8") as handle:
        handle.write("student_id,subject,marks\n"
                     "101,Math,90\n101,Physics,80\n101,English,85\n"
                     "102,Math,95\n102,Physics,70\n"
                     "103,Math,35\n103,Math,50\n"          # duplicate subject -> rejected
                     "999,Math,80\n"                       # unknown ID -> rejected
                     "102,English,120\n")                  # invalid marks -> rejected


def main():
    create_sample_data()
    try:
        students = load_students("students.csv")
        scores, rejected = load_scores("scores.csv", students)
    except (OSError, ValueError) as err:
        print("Could not read input files:", err)
        return
    report = build_report(students, scores)
    write_report(report)
    for row in report:
        print(row)
    print("Rejected score rows:")
    for row in rejected:
        print("  ", row)


if __name__ == "__main__":
    main()


{'student_id': '101', 'name': 'Ali', 'subjects': 3, 'average': 85.0, 'rank': 1, 'grade': 'A'}
{'student_id': '102', 'name': 'Sara', 'subjects': 2, 'average': 82.5, 'rank': 2, 'grade': 'B'}
{'student_id': '103', 'name': 'Bilal', 'subjects': 1, 'average': 35.0, 'rank': 3, 'grade': 'F'}
Rejected score rows:
   {'student_id': '103', 'subject': 'Math', 'marks': '50', 'reason': 'duplicate student-subject entry'}
   {'student_id': '999', 'subject': 'Math', 'marks': '80', 'reason': 'unknown student id'}
   {'student_id': '102', 'subject': 'English', 'marks': '120', 'reason': 'marks must be between 0 and 100'}


Task 12 | Custom Iterable Range Toolkit<br>
Concepts: Iterators, generators, dunder methods

In [32]:
class ReversibleRange:
    """Works like range(start, stop, step) but is written by hand (no list is built)."""

    def __init__(self, start, stop, step=1):
        for label, value in (("start", start), ("stop", stop), ("step", step)):
            if isinstance(value, bool) or not isinstance(value, int):
                raise TypeError(f"{label} must be an integer")
        if step == 0:
            raise ValueError("step must not be zero")
        self.start, self.stop, self.step = start, stop, step

    def __len__(self):
        if self.step > 0:
            span = self.stop - self.start
            return max(0, (span + self.step - 1) // self.step)
        span = self.start - self.stop
        return max(0, (span - self.step - 1) // (-self.step))

    def __iter__(self):
        current = self.start
        if self.step > 0:
            while current < self.stop:
                yield current
                current += self.step
        else:
            while current > self.stop:
                yield current
                current += self.step

    def __reversed__(self):
        count = len(self)
        for i in range(count - 1, -1, -1):          # last item first
            yield self.start + i * self.step

    def __contains__(self, value):
        if isinstance(value, bool) or not isinstance(value, int):
            return False
        if self.step > 0:
            inside = self.start <= value < self.stop
        else:
            inside = self.stop < value <= self.start
        return inside and (value - self.start) % self.step == 0

    def __repr__(self):
        return f"ReversibleRange({self.start}, {self.stop}, {self.step})"


if __name__ == "__main__":
    r = ReversibleRange(2, 11, 3)
    print(list(r), list(reversed(r)), len(r), 5 in r, 6 in r)       # [2,5,8] [8,5,2] 3 True False
    n = ReversibleRange(10, 0, -3)
    print(list(n), list(reversed(n)), len(n), 7 in n, 0 in n)       # [10,7,4,1] [1,4,7,10] 4 True False
    print(list(ReversibleRange(5, 5)), len(ReversibleRange(5, 1)))  # [] 0
    # compare with the real range
    for a, b, c in [(0, 10, 1), (0, 10, 4), (10, 0, -2), (3, -8, -5), (-5, 5, 3)]:
        mine, real = ReversibleRange(a, b, c), range(a, b, c)
        assert list(mine) == list(real) and list(reversed(mine)) == list(reversed(real))
        assert len(mine) == len(real) and all((x in mine) == (x in real) for x in range(-12, 13))
    print("Matches Python's range")


[2, 5, 8] [8, 5, 2] 3 True False
[10, 7, 4, 1] [1, 4, 7, 10] 4 True False
[] 0
Matches Python's range


Task 13 | Booking Conflict Scheduler<br>
Concepts: Datetime, sorting, interval algorithms

In [33]:
from dataclasses import dataclass
from datetime import datetime


@dataclass(frozen=True)
class Booking:
    booking_id: str
    room: str
    start: datetime
    end: datetime


class Scheduler:
    def __init__(self):
        self._bookings = {}      # room -> list of Booking sorted by start time
        self._ids = set()

    def book(self, booking_id, room, start, end):
        """Try to book a room.

        Returns (True, []) if accepted, or (False, [conflicting bookings]) if rejected.
        Raises ValueError for invalid input (end <= start, duplicate ID, bad types).
        """
        if not isinstance(start, datetime) or not isinstance(end, datetime):
            raise ValueError("start and end must be datetime objects")
        if not isinstance(room, str) or not room.strip():
            raise ValueError("room must be a non-empty string")
        if end <= start:
            raise ValueError("end must be after start")
        if booking_id in self._ids:
            raise ValueError(f"Duplicate booking id: {booking_id}")

        existing = self._bookings.setdefault(room, [])
        # two intervals overlap only if each starts before the other ends,
        # so back-to-back bookings (end == start) are allowed
        conflicts = [b for b in existing if start < b.end and b.start < end]
        if conflicts:
            return False, conflicts

        existing.append(Booking(booking_id, room, start, end))
        existing.sort(key=lambda b: b.start)
        self._ids.add(booking_id)
        return True, []

    def bookings_for(self, room):
        return list(self._bookings.get(room, []))


if __name__ == "__main__":
    def at(hour, minute=0):
        return datetime(2026, 11, 7, hour, minute)

    scheduler = Scheduler()
    requests = [("B1", "A", at(9), at(10)), ("B2", "A", at(10), at(11)),
                ("B3", "A", at(9, 30), at(10, 30)), ("B4", "B", at(9, 30), at(10, 30)),
                ("B1", "B", at(12), at(13)), ("B5", "A", at(12), at(11))]
    for booking_id, room, start, end in requests:
        try:
            ok, conflicts = scheduler.book(booking_id, room, start, end)
        except ValueError as err:
            print(booking_id, "invalid:", err)
            continue
        print(booking_id, "accepted" if ok else f"rejected, conflicts with {[c.booking_id for c in conflicts]}")


B1 accepted
B2 accepted
B3 rejected, conflicts with ['B1', 'B2']
B4 accepted
B1 invalid: Duplicate booking id: B1
B5 invalid: end must be after start


Task 14 | Command-Line Contact Manager<br>
Concepts: Modules, JSON, CRUD, validation

In [37]:
import json
import os
import re
import shutil
import time

DATA_FILE = "contacts.json"
FIELDS = {"id", "name", "phone", "email"}


# ======================= storage layer =======================
class StorageError(Exception):
    pass


def load_contacts(path=DATA_FILE):
    """Return the list of contacts. Missing file -> empty list.

    A corrupt file is never overwritten: it is copied to a backup and an error is raised.
    """
    if not os.path.exists(path):
        return []
    try:
        with open(path, encoding="utf-8") as handle:
            data = json.load(handle)
        if not isinstance(data, list) or not all(
                isinstance(c, dict) and FIELDS <= c.keys() for c in data):
            raise ValueError("unexpected structure")
        return data
    except (ValueError, OSError) as err:
        backup = f"{path}.corrupt-{int(time.time())}"
        try:
            shutil.copy2(path, backup)
        except OSError:
            backup = "(backup failed)"
        raise StorageError(f"{path} could not be read ({err}). Original kept, copy: {backup}")


def save_contacts(contacts, path=DATA_FILE):
    """Write to a temp file first, then replace - a crash cannot leave a half-written file."""
    temp = path + ".tmp"
    with open(temp, "w", encoding="utf-8") as handle:
        json.dump(contacts, handle, indent=2)
    os.replace(temp, path)


# ======================= business logic =======================
def validate_contact(name, phone, email):
    if not name or not name.strip():
        raise ValueError("Name cannot be empty")
    if not re.fullmatch(r"\+?[0-9][0-9\- ]{5,18}[0-9]", phone.strip()):
        raise ValueError("Phone must have 7-20 digits (spaces, '-' and leading '+' allowed)")
    if not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", email.strip()):
        raise ValueError("Email is not valid")


def add_contact(contacts, name, phone, email):
    validate_contact(name, phone, email)
    new_id = max((c["id"] for c in contacts), default=0) + 1     # unique ID
    contact = {"id": new_id, "name": name.strip(), "phone": phone.strip(), "email": email.strip()}
    contacts.append(contact)
    return contact


def search_contacts(contacts, query):
    query = query.strip().lower()
    return [c for c in contacts if query in c["name"].lower()]    # case-insensitive


def find_contact(contacts, contact_id):
    for contact in contacts:
        if contact["id"] == contact_id:
            return contact
    raise KeyError(f"No contact with id {contact_id}")


def update_contact(contacts, contact_id, name=None, phone=None, email=None):
    contact = find_contact(contacts, contact_id)
    new_name = contact["name"] if not name else name
    new_phone = contact["phone"] if not phone else phone
    new_email = contact["email"] if not email else email
    validate_contact(new_name, new_phone, new_email)
    contact.update(name=new_name.strip(), phone=new_phone.strip(), email=new_email.strip())
    return contact


def delete_contact(contacts, contact_id):
    contact = find_contact(contacts, contact_id)
    contacts.remove(contact)
    return contact


# ======================= menu (CLI) =======================
def show(contacts):
    if not contacts:
        print("  (no contacts)")
    for c in contacts:
        print(f"  [{c['id']}] {c['name']} | {c['phone']} | {c['email']}")


def ask_id(prompt):
    try:
        return int(input(prompt).strip())
    except ValueError:
        raise ValueError("ID must be a number")


def main(path=DATA_FILE):
    try:
        contacts = load_contacts(path)
    except StorageError as err:
        print("ERROR:", err)
        print("Exiting without changing anything.")
        return

    menu = "\n1) Add  2) Search  3) Update  4) Delete  5) List  0) Exit"
    while True:
        print(menu)
        choice = input("Choose: ").strip()
        try:
            if choice == "1":
                contact = add_contact(contacts, input("Name: "), input("Phone: "), input("Email: "))
                save_contacts(contacts, path)
                print("Added with ID", contact["id"])
            elif choice == "2":
                show(search_contacts(contacts, input("Search name: ")))
            elif choice == "3":
                contact_id = ask_id("ID to update: ")
                print("(leave blank to keep the current value)")
                update_contact(contacts, contact_id, input("Name: "), input("Phone: "), input("Email: "))
                save_contacts(contacts, path)
                print("Updated.")
            elif choice == "4":
                removed = delete_contact(contacts, ask_id("ID to delete: "))
                save_contacts(contacts, path)
                print("Deleted", removed["name"])
            elif choice == "5":
                show(contacts)
            elif choice == "0":
                print("Goodbye!")
                break
            else:
                print("Invalid choice.")
        except (ValueError, KeyError) as err:
            print("Error:", err.args[0] if err.args else err)
        except OSError as err:
            print("Could not save:", err)


if __name__ == "__main__":
    main()



1) Add  2) Search  3) Update  4) Delete  5) List  0) Exit
Choose: 1
Name: joe
Phone: 12312312332
Email: joe@rww.com
Added with ID 1

1) Add  2) Search  3) Update  4) Delete  5) List  0) Exit
Choose: 5
  [1] joe | 12312312332 | joe@rww.com

1) Add  2) Search  3) Update  4) Delete  5) List  0) Exit
Choose: 0
Goodbye!


Task 15 | Integrated Order Fulfillment System<br>
Concepts: OOP, composition, files, exceptions, algorithms


In [36]:
import json
import os
import sys
import tempfile
import unittest
import uuid
from dataclasses import dataclass
from datetime import datetime


class OutOfStockError(Exception):
    pass


class OrderPersistenceError(Exception):
    pass


# ======================= classes =======================
@dataclass(frozen=True)
class Product:
    product_id: str
    name: str
    price: float


class Inventory:
    def __init__(self):
        self._products = {}
        self._stock = {}

    def add_product(self, product, stock):
        if stock < 0:
            raise ValueError("Stock cannot be negative")
        self._products[product.product_id] = product
        self._stock[product.product_id] = stock

    @classmethod
    def from_json(cls, path):
        """Catalog format: [{"id": "P1", "name": "Pen", "price": 100, "stock": 5}, ...]"""
        with open(path, encoding="utf-8") as handle:
            items = json.load(handle)
        inventory = cls()
        for item in items:
            inventory.add_product(Product(item["id"], item["name"], float(item["price"])),
                                  int(item["stock"]))
        return inventory

    def get_product(self, product_id):
        if product_id not in self._products:
            raise KeyError(f"Unknown product: {product_id}")
        return self._products[product_id]

    def stock_of(self, product_id):
        self.get_product(product_id)
        return self._stock[product_id]

    def check_available(self, items):
        """Validate EVERY item first (items = {product_id: quantity}). Changes nothing."""
        for product_id, quantity in items.items():
            if self.stock_of(product_id) < quantity:
                raise OutOfStockError(
                    f"Not enough stock for {product_id}: wanted {quantity}, "
                    f"have {self._stock[product_id]}")

    def remove(self, items):
        for product_id, quantity in items.items():
            self._stock[product_id] -= quantity

    def restore(self, items):
        for product_id, quantity in items.items():
            self._stock[product_id] += quantity


class Cart:
    def __init__(self, inventory):
        self._inventory = inventory
        self.items = {}               # product_id -> quantity
        self.discount_percent = 0

    def add(self, product_id, quantity):
        if isinstance(quantity, bool) or not isinstance(quantity, int) or quantity <= 0:
            raise ValueError(f"Quantity must be a positive whole number, got {quantity!r}")
        self._inventory.get_product(product_id)       # raises KeyError if unknown
        self.items[product_id] = self.items.get(product_id, 0) + quantity

    def apply_discount(self, percent):
        if isinstance(percent, bool) or not isinstance(percent, (int, float)) \
                or not 0 <= percent <= 100:
            raise ValueError("Discount must be between 0 and 100 percent")
        self.discount_percent = percent

    def subtotal(self):
        return sum(self._inventory.get_product(pid).price * qty for pid, qty in self.items.items())

    def total(self):
        return round(self.subtotal() * (100 - self.discount_percent) / 100, 2)

    def is_empty(self):
        return not self.items

    def clear(self):
        self.items = {}
        self.discount_percent = 0


@dataclass
class Order:
    order_id: str
    lines: list
    discount_percent: float
    total: float
    created_at: str

    def to_dict(self):
        return {"order_id": self.order_id, "lines": self.lines,
                "discount_percent": self.discount_percent, "total": self.total,
                "created_at": self.created_at}


# ======================= persistence + placing orders =======================
def save_order(order, path):
    orders = []
    if os.path.exists(path):
        with open(path, encoding="utf-8") as handle:
            orders = json.load(handle)           # corrupt file -> exception -> rollback
    orders.append(order.to_dict())
    temp = path + ".tmp"
    with open(temp, "w", encoding="utf-8") as handle:
        json.dump(orders, handle, indent=2)
    os.replace(temp, path)


def place_order(cart, inventory, orders_path):
    if cart.is_empty():
        raise ValueError("Cannot place an order with an empty cart")

    inventory.check_available(cart.items)         # 1) validate everything first

    lines = []
    for product_id, quantity in cart.items.items():
        product = inventory.get_product(product_id)
        lines.append({"product_id": product_id, "name": product.name,
                      "unit_price": product.price, "quantity": quantity})
    order = Order(uuid.uuid4().hex[:12].upper(), lines, cart.discount_percent,
                  cart.total(), datetime.now().isoformat(timespec="seconds"))

    purchased = dict(cart.items)
    inventory.remove(purchased)                   # 2) update stock
    try:
        save_order(order, orders_path)            # 3) persist
    except Exception as err:
        inventory.restore(purchased)              # roll back stock if saving fails
        raise OrderPersistenceError(f"Order not saved, stock restored: {err}") from err

    cart.clear()
    return order


# ======================= unit tests =======================
class OrderSystemTests(unittest.TestCase):
    def setUp(self):
        self.temp_dir = tempfile.TemporaryDirectory()
        self.orders_path = os.path.join(self.temp_dir.name, "orders.json")
        self.inventory = Inventory()
        self.inventory.add_product(Product("P1", "Notebook", 100), 5)
        self.inventory.add_product(Product("P2", "Pen", 10), 1)

    def tearDown(self):
        self.temp_dir.cleanup()

    def test_successful_order_with_discount(self):
        cart = Cart(self.inventory)
        cart.add("P1", 2)
        cart.apply_discount(10)
        order = place_order(cart, self.inventory, self.orders_path)
        self.assertEqual(order.total, 180)
        self.assertEqual(self.inventory.stock_of("P1"), 3)
        with open(self.orders_path, encoding="utf-8") as handle:
            self.assertEqual(len(json.load(handle)), 1)

    def test_out_of_stock_changes_nothing(self):
        cart = Cart(self.inventory)
        cart.add("P1", 2)
        cart.add("P2", 5)                          # only 1 in stock
        with self.assertRaises(OutOfStockError):
            place_order(cart, self.inventory, self.orders_path)
        self.assertEqual(self.inventory.stock_of("P1"), 5)   # nothing was deducted
        self.assertFalse(os.path.exists(self.orders_path))

    def test_invalid_quantity_rejected(self):
        cart = Cart(self.inventory)
        for bad in (0, -1, 1.5, "2", True):
            with self.assertRaises(ValueError):
                cart.add("P1", bad)

    def test_rollback_when_saving_fails(self):
        cart = Cart(self.inventory)
        cart.add("P1", 2)
        with self.assertRaises(OrderPersistenceError):
            place_order(cart, self.inventory, self.temp_dir.name)   # a folder, not a file
        self.assertEqual(self.inventory.stock_of("P1"), 5)          # stock restored

    def test_order_ids_are_unique(self):
        ids = set()
        for _ in range(3):
            cart = Cart(self.inventory)
            cart.add("P1", 1)
            ids.add(place_order(cart, self.inventory, self.orders_path).order_id)
        self.assertEqual(len(ids), 3)


# ======================= demo =======================
def demo():
    with open("catalog.json", "w", encoding="utf-8") as handle:
        json.dump([{"id": "P1", "name": "Notebook", "price": 100, "stock": 5},
                   {"id": "P2", "name": "Pen", "price": 10, "stock": 20}], handle, indent=2)

    inventory = Inventory.from_json("catalog.json")
    cart = Cart(inventory)
    cart.add("P1", 2)
    cart.apply_discount(10)
    order = place_order(cart, inventory, "orders.json")
    print("Order", order.order_id, "total =", order.total)        # 180.0
    print("P1 stock now:", inventory.stock_of("P1"))              # 3

    cart.add("P1", 10)
    try:
        place_order(cart, inventory, "orders.json")
    except OutOfStockError as err:
        print("Rejected:", err)


if __name__ == "__main__":
    if len(sys.argv) > 1 and sys.argv[1] == "test":
        unittest.main(argv=[sys.argv[0]], verbosity=2)
    else:
        demo()


Order 2E9E029A84A0 total = 180.0
P1 stock now: 3
Rejected: Not enough stock for P1: wanted 10, have 3
